# Ablation Study
Compare full features vs without spatial vs without engineered features.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

df = pd.read_csv('../datasets/processed/merged_soil_data.csv')

FULL_FEATURES = [
    'N','P','K','ph','ec','organic_carbon','moisture',
    'temperature','humidity','rainfall',
    'soil_quality_index','fertility_score',
    'lat_lon_cluster','region_code','season_encoded'
]

WITHOUT_SPATIAL = [
    'N','P','K','ph','ec','organic_carbon','moisture',
    'temperature','humidity','rainfall'
]

WITHOUT_ENGINEERED = [
    'N','P','K','ph','ec','organic_carbon','moisture',
    'temperature','humidity','rainfall',
    'lat_lon_cluster','region_code'
]

configs = {
    'Full Feature Set (Ours)': FULL_FEATURES,
    'Without Spatial Features': WITHOUT_SPATIAL,
    'Without Engineered Features': WITHOUT_ENGINEERED
}

results = {}
for name, cols in configs.items():
    available = [c for c in cols if c in df.columns]
    X = df[available].values
    y = df['crop'].values
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))
    results[name] = acc
    print(f'{name}: {acc:.4f}')

plt.figure(figsize=(10,5))
plt.bar(results.keys(), [v*100 for v in results.values()], color=['green','orange','red'])
plt.title('Ablation Study: Impact of Feature Engineering on Crop Accuracy')
plt.ylabel('Accuracy (%)')
plt.ylim(80, 100)
plt.tight_layout()
plt.savefig('../reports/ablation_study.png')
plt.show()